# Kapitel 10: Aufmerksamkeit ist alles, was Sie brauchen

> "Attention is All You Need." — **Vaswani et al.**, Google Research, 2017

---

## Was Sie lernen werden

- Warum statische Einbettungen nicht ausreichen (das "Bank" in "Flussufer" vs. "Sparkasse")
- Was Aufmerksamkeit (Attention) auf konzeptioneller Ebene IST (Tokens, die einander betrachten)
- Wie man Selbstaufmerksamkeit (Self-Attention) Schritt für Schritt aus den Konzepten Query, Key und Value aufbaut
- Warum wir Aufmerksamkeits-Scores skalieren und zukünftige Tokens maskieren
- Wie man von einzelner zu mehrköpfiger Aufmerksamkeit (Multi-Head Attention) effizient übergeht
- Wie man Aufmerksamkeit mit Feedforward-Netzwerken zu vollständigen Transformer-Blöcken kombiniert
- Wie man visualisiert, welche Aufmerksamkeitsmuster entstehen

---

## Einrichtung

Zunächst installieren wir die erforderlichen Pakete:

In [ ]:
# Erforderliche Pakete installieren
!pip install -q torch transformers matplotlib

In [ ]:
# ===== IMPORTS =====
import torch                     # PyTorch: Tensor-Operationen
import torch.nn as nn            # Neuronale Netzwerkschichten
import torch.nn.functional as F  # Mathematische Funktionen (softmax, gelu)
import math                      # Für sqrt in der Aufmerksamkeitsskalierung
import matplotlib.pyplot as plt  # Visualisierung
import numpy as np               # Array-Operationen

# Wichtige Bausteine, die wir verwenden werden:
# - nn.Linear(in, out): Matrixmultiplikationsschicht (lernt Gewichte)
# - F.softmax(x, dim): Wandelt Scores in Wahrscheinlichkeiten um (Summe = 1)
# - @ Operator: Matrixmultiplikation (identisch mit torch.matmul)

## 1. Warum statische Einbettungen nicht ausreichen

Das Problem: Einbettungen aus Kapitel 9 sind **statisch** – jedes Token erhält denselben Vektor unabhängig vom Kontext.

In [ ]:
# Simuliere Ausgabe aus GPT2Embeddings aus Kapitel 9
batch_size = 2
seq_len = 6
embed_dim = 768

# Einbettungen aus Kap. 9 (zufällig für dieses Beispiel, aber stellen Sie sich vor, sie wären echt)
embeddings = torch.randn(batch_size, seq_len, embed_dim)
print(f"Form der Eingabe-Einbettungen: {embeddings.shape}")
# Erwartete Ausgabe: torch.Size([2, 6, 768])

# Dies sind STATISCHE Einbettungen aus Kap. 9
# Unser Ziel: Sie in KONTEXTBEWUSSTE Einbettungen umwandeln

print("\nDas Wort 'Bank' bekommt immer denselben Vektor:")
print("- 'Flussufer' → dieselbe Einbettung")
print("- 'Sparkasse' → dieselbe Einbettung")
print("- Aber sie bedeuten völlig unterschiedliche Dinge!")

## 2. Selbstaufmerksamkeit Schritt für Schritt aufbauen

Bauen wir Aufmerksamkeit schrittweise auf und zeigen die Formen bei jedem Schritt.

**Die Intuition:** Jedes Token fragt "Welche anderen Tokens sind relevant, um mich zu verstehen?"
- **Abfrage (Query, Q)**: "Wonach suche ich?"
- **Schlüssel (Key, K)**: "Was biete ich an?"
- **Wert (Value, V)**: "Meine Informationen zum Teilen"

Denken Sie daran wie an eine Suchmaschine: Query ist Ihre Suche, Keys sind die Titel/Tags von Dokumenten, Values sind der eigentliche Inhalt.

### Schritt 1: Query-, Key- und Value-Projektionen erstellen

**Was ist `nn.Linear(in_dim, out_dim)`?**
- Erstellt eine Gewichtsmatrix der Form (in_dim, out_dim)
- Wenn Sie eine Eingabe durchleiten: `output = input @ weight`
- Diese Gewichte sind "lernbar" – sie werden während des Trainings aktualisiert

In [ ]:
# Verwenden wir zur Klarheit eine kleinere Dimension
d_model = 768  # Von Einbettungen (Kap. 9)
d_k = 64       # Dimension für Q, K, V (typisch: d_model / num_heads)

# Projektionsschichten erstellen (diese haben lernbare Parameter!)
W_q = nn.Linear(d_model, d_k, bias=False)  # Query-Projektion
W_k = nn.Linear(d_model, d_k, bias=False)  # Key-Projektion
W_v = nn.Linear(d_model, d_k, bias=False)  # Value-Projektion

# Projiziere Einbettungen auf Q, K, V
# Warum Linear? Es lernt die beste Transformation für jede Rolle
Q = W_q(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
K = W_k(embeddings)  # (batch, seq, d_k) = (2, 6, 64)
V = W_v(embeddings)  # (batch, seq, d_k) = (2, 6, 64)

print(f"Q-Form: {Q.shape}")  # Erwartet: torch.Size([2, 6, 64])
print(f"K-Form: {K.shape}")  # Erwartet: torch.Size([2, 6, 64])
print(f"V-Form: {V.shape}")  # Erwartet: torch.Size([2, 6, 64])

print("\nJedes Token hat jetzt:")
print("- Q-Vektor (64 Dims): 'Wonach ich suche'")
print("- K-Vektor (64 Dims): 'Was ich anbiete'")
print("- V-Vektor (64 Dims): 'Meine Informationen zum Teilen'")

### Schritt 2: Aufmerksamkeits-Scores berechnen (Q · K^T)

In [ ]:
# Berechne Aufmerksamkeits-Scores: Q @ K^T
# Wir müssen K transponieren, damit die Dimensionen für matmul übereinstimmen

# Q-Form: (batch, seq, d_k) = (2, 6, 64)
# K-Form: (batch, seq, d_k) = (2, 6, 64)

# K.transpose(-2, -1) vertauscht die letzten beiden Dimensionen:
# Negative Indizes: -1 = letzte Dim, -2 = vorletzte Dim
# Also K geht von (2, 6, 64) → (2, 64, 6)

# Matrixmultiplikation: (2, 6, 64) @ (2, 64, 6) → (2, 6, 6)
scores = Q @ K.transpose(-2, -1)

print(f"Q-Form: {Q.shape}")
print(f"K-Form: {K.shape}")
print(f"K transponiert Form: {K.transpose(-2, -1).shape}")
print(f"Aufmerksamkeits-Scores Form: {scores.shape}")
# Erwartet: torch.Size([2, 6, 6])

print(f"\nScores für erstes Item im Batch:")
print(scores[0])
print("\n6×6-Matrix, wobei Eintrag [i,j] = wie viel Token i auf Token j achtet")

### Schritt 3: Die Scores skalieren

In [ ]:
# Skaliere durch sqrt(Dimension)
# Warum sqrt? Mathematischer Beweis zeigt, dass dies die Varianz stabil hält
scores = scores / math.sqrt(d_k)

print(f"Skalierte Scores Form: {scores.shape}")  # Immer noch (2, 6, 6)
print(f"\nVor der Skalierung könnte der Score-Bereich sein: ±{d_k}")
print(f"Nach Skalierung durch sqrt({d_k}) = {math.sqrt(d_k):.2f}, ist der Bereich ungefähr: ±8")

print("\nWarum das wichtig ist: Ohne Skalierung würde hochdimensionale Aufmerksamkeit")
print("fast das gesamte Gewicht auf ein Token legen und den Nutzen verlieren,")
print("auf mehrere Tokens zu achten.")

### Schritt 4: Softmax anwenden, um Aufmerksamkeitsgewichte zu erhalten

In [ ]:
# Wende Softmax über die letzte Dimension an (über Keys)
# Dies bewirkt, dass jede Zeile (jede Query) zu 1 summiert
attn_weights = F.softmax(scores, dim=-1)

print(f"Aufmerksamkeitsgewichte Form: {attn_weights.shape}")  # Erwartet: (2, 6, 6)
print(f"\nAufmerksamkeitsgewichte für Token 0 (erster Batch):")
print(attn_weights[0, 0])
# Beispielausgabe: tensor([0.15, 0.20, 0.30, 0.18, 0.10, 0.07])
# Diese summieren sich zu 1.0!

print(f"\nSumme der Gewichte für Token 0: {attn_weights[0, 0].sum().item():.4f}")
# Erwartet: Summe der Gewichte für Token 0: 1.0000

### Schritt 5: Gewichtete Summe der Werte

In [ ]:
# Aufmerksamkeitsgewichte: (batch, seq, seq) = (2, 6, 6)
# Werte:                   (batch, seq, d_k)  = (2, 6, 64)
# Wir wollen:              (batch, seq, d_k)  = (2, 6, 64)

output = attn_weights @ V

print(f"Ausgabe-Form: {output.shape}")  # Erwartet: torch.Size([2, 6, 64])

print(f"\nOriginale Einbettung für Token 0 (erste 10 Dims):")
print(embeddings[0, 0, :10])

print(f"\nAusgabe nach Aufmerksamkeit für Token 0 (erste 10 Dims):")
print(output[0, 0, :10])
print("\nUnterschiedliche Werte! Dieses Token hat Kontext integriert")

### Vollständige Aufmerksamkeitsfunktion

Verpacken wir die 5 Schritte in eine Funktion:

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Berechne skalierte Punktprodukt-Aufmerksamkeit.
    
    Args:
        Q: Abfragen (batch, seq, d_k)
        K: Schlüssel (batch, seq, d_k)
        V: Werte     (batch, seq, d_k)
        mask: Optionale Maske (batch, seq, seq)
    
    Returns:
        output: Aufmerksamkeitsausgabe (batch, seq, d_k)
        attn_weights: Aufmerksamkeitsgewichte (batch, seq, seq)
    """
    d_k = Q.size(-1)  # Dimension der Abfragen/Schlüssel abrufen
    
    # Schritt 1: Scores Q @ K^T berechnen
    scores = Q @ K.transpose(-2, -1)  # (batch, seq, seq)
    
    # Schritt 2: Durch sqrt(d_k) skalieren
    scores = scores / math.sqrt(d_k)
    
    # Schritt 3: Maske anwenden, falls vorhanden
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Schritt 4: Softmax für Aufmerksamkeitsgewichte
    attn_weights = F.softmax(scores, dim=-1)  # (batch, seq, seq)
    
    # Schritt 5: Gewichtete Summe der Werte
    output = attn_weights @ V  # (batch, seq, d_k)
    
    return output, attn_weights


# Teste es
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Ausgabe-Form: {output.shape}")  # Erwartet: (2, 6, 64)
print(f"Aufmerksamkeitsgewichte Form: {attn_weights.shape}")  # Erwartet: (2, 6, 6)
print(f"\nAufmerksamkeitsverteilung des ersten Tokens:")
print(attn_weights[0, 0])

## 3. Kausale Maskierung für autoregressive Generierung

### Das Schummelproblem

Ohne Maskierung kann das Modell bei der Verarbeitung von "Katze" auf ALLE Tokens achten – einschließlich zukünftiger! Das ist Schummeln während des Trainings.

In [ ]:
def create_causal_mask(seq_len):
    """
    Erstelle eine kausale Maske: oberes Dreieck ist False (blockieren), unteres ist True (erlauben).
    
    Returns:
        mask: (seq_len, seq_len) boolean Tensor
    """
    # torch.tril erstellt eine untere Dreiecksmatrix
    # 1en unterhalb der Diagonale (einschließlich Diagonale), 0en darüber
    mask = torch.tril(torch.ones(seq_len, seq_len))
    
    return mask


# Beispiel mit seq_len = 6
mask = create_causal_mask(6)
print("Kausale Maske (1 = erlauben, 0 = blockieren):")
print(mask)

print("\nLesen der Maske:")
print("- Zeile 0 (Token 0): Kann nur auf Spalte 0 achten")
print("- Zeile 2 (Token 2): Kann auf Spalten 0, 1, 2 achten")
print("- Zeile 5 (Token 5): Kann auf alle Spalten 0-5 achten")

### Die Maske anwenden

In [ ]:
# Test mit kausaler Maske
Q_test = torch.randn(2, 6, 64)  # (batch, seq, d_k)
K_test = torch.randn(2, 6, 64)
V_test = torch.randn(2, 6, 64)

# Kausale Maske erstellen
causal_mask = create_causal_mask(6)  # (seq, seq)

# Aufmerksamkeit mit Maske anwenden
output_masked, attn_weights_masked = scaled_dot_product_attention(
    Q_test, K_test, V_test, mask=causal_mask
)

print("Aufmerksamkeitsgewichte MIT kausaler Maske (erstes Item im Batch):")
print(attn_weights_masked[0])

print("\nBeachten Sie: Oberes Dreieck ist alles Null! Keine Aufmerksamkeit auf die Zukunft!")

## 4. Mehrköpfige Aufmerksamkeit (Multi-Head Attention)

### Warum mehrere Köpfe?

Ein Aufmerksamkeitskopf kann nur EINE Art von Beziehung gleichzeitig erfassen. Mehrere Köpfe ermöglichen es dem Modell, verschiedene Muster gleichzeitig zu lernen:

| Kopf | Was er lernen könnte |
|------|---------------------|
| Kopf 1 | Subjekt-Verb-Beziehungen ("Katze" → "saß") |
| Kopf 2 | Adjektiv-Substantiv-Verbindungen ("faul" → "Hund") |
| Kopf 3 | Nahegelegene Wortmuster (lokaler Kontext) |
| Kopf 4 | Langstrecken-Abhängigkeiten (Pronomen-Auflösung) |

**Wichtige Erkenntnis:** 12 Köpfe mit je 64 Dims = 768 Gesamt-Dims = gleich wie 1 großer Kopf!
Keine zusätzlichen Parameter – nur verschiedene Perspektiven.

### Effiziente Implementierung

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Effiziente mehrköpfige Aufmerksamkeit (gruppiert alle Köpfe zusammen).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64
        
        # Kombinierte QKV-Projektion (3x effizienter als separate!)
        # Warum 3 * d_model? Weil wir gleichzeitig auf Q, K, V projizieren
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        
        # Ausgabeprojektion
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        # Dropout auf Aufmerksamkeitsgewichten
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape
        
        # ===== Schritt 1: Auf Q, K, V projizieren (alles auf einmal!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)
        
        # ===== Schritt 2: In Q, K, V aufteilen und für Multi-Head umformen =====
        # Umformen zu (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)
        
        # Permutieren zu (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        
        # In Q, K, V aufteilen: jedes ist (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # ===== Schritt 3: Skalierte Punktprodukt-Aufmerksamkeit (über Köpfe gebatched) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)
        
        # Kausale Maske anwenden, falls vorhanden
        if mask is not None:
            # Maske für Köpfe erweitern: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)
        
        # Gewichtete Summe der Werte
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)
        
        # ===== Schritt 4: Köpfe verketten =====
        # Transponieren zu (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)
        
        # Umformen zu (batch, seq, d_model) – dies verkettet die Köpfe
        attn_output = attn_output.reshape(batch, seq, d_model)
        
        # ===== Schritt 5: Finale Projektion =====
        output = self.out_proj(attn_output)
        
        return output, attn_weights


# Teste es
mha = MultiHeadAttention(d_model=768, num_heads=12, dropout=0.1)
embeddings_test = torch.randn(2, 6, 768)
mask_test = create_causal_mask(6)

output_mha, attn_weights_mha = mha(embeddings_test, mask_test)

print(f"Eingabe-Form:  {embeddings_test.shape}")     # Erwartet: (2, 6, 768)
print(f"Ausgabe-Form: {output_mha.shape}")          # Erwartet: (2, 6, 768)
print(f"Aufmerksamkeitsgewichte Form: {attn_weights_mha.shape}")  # Erwartet: (2, 12, 6, 6)
print("                                                         ^^ 12 Köpfe!")

## 5. Vollständige Transformer-Blöcke

### Feedforward-Netzwerk

Das Feedforward-Netzwerk erweitert die Dimension (768 → 3072), wendet eine Nichtlinearität an und komprimiert dann zurück (3072 → 768).

**Was ist GELU?**
- GELU (Gaussian Error Linear Unit) ist eine Aktivierungsfunktion
- Wie ReLU, aber glatter – hat keinen harten Cutoff bei Null
- Wird in GPT-2, BERT und den meisten modernen Transformers verwendet
- Intuition: "steuert", wie viel Signal basierend auf der Eingangsgröße durchgelassen wird

In [ ]:
class FeedForward(nn.Module):
    """
    Positionsweises Feedforward-Netzwerk.
    Wird auf jede Position unabhängig angewendet (gleiche Gewichte für alle Positionen).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Modelldimension (768 für GPT-2 small)
            d_ff: Feedforward-Dimension (typisch 4 * d_model = 3072)
            dropout: Dropout-Wahrscheinlichkeit
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff) – erweitern
        x = F.gelu(x)          # Nichtlinearität
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model) – zurück projizieren
        return x


# Teste es
ffn = FeedForward(d_model=768, d_ff=3072, dropout=0.1)
x_test = torch.randn(2, 6, 768)
output_ffn = ffn(x_test)

print(f"Eingabe-Form:  {x_test.shape}")      # Erwartet: (2, 6, 768)
print(f"Ausgabe-Form: {output_ffn.shape}")  # Erwartet: (2, 6, 768)

### Vollständiger Transformer-Block (Pre-Norm-Stil)

**Pre-Norm vs. Post-Norm:**
- **Post-Norm** (ursprünglicher Transformer): Schichtnormalisierung (LayerNorm) NACH jeder Unterschicht
- **Pre-Norm** (GPT-2, modern): Schichtnormalisierung VOR jeder Unterschicht

Warum Pre-Norm? Es macht das Training für tiefe Netzwerke (12+ Schichten) stabiler. Die Gradienten fließen glatter durch die residualen Verbindungen.

**Residuale Verbindungen:** `output = x + sublayer(x)`
- Erstellt "Autobahnen" für Gradienten, um rückwärts zu fließen
- Ohne residuale Verbindungen verschwinden Gradienten in tiefen Netzwerken

In [ ]:
class TransformerBlock(nn.Module):
    """
    Vollständiger Transformer-Block mit mehrköpfiger Aufmerksamkeit, Feedforward,
    residualen Verbindungen und Schichtnormalisierung (Pre-Norm-Stil wie GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Modelldimension (768)
            num_heads: Anzahl der Aufmerksamkeitsköpfe (12)
            d_ff: Feedforward-Dimension (3072 = 4 * d_model)
            dropout: Dropout-Wahrscheinlichkeit
        """
        super().__init__()
        
        # Schichtnormalisierung (vor jeder Unterschicht)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # Mehrköpfige Aufmerksamkeit
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Feedforward-Netzwerk
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        # Dropout (nach jeder Unterschicht angewendet)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: Eingabe-Einbettungen (batch, seq, d_model)
            mask: Kausale Maske (seq, seq) oder (1, 1, seq, seq)
        
        Returns:
            x: Ausgabe-Einbettungen (batch, seq, d_model)
            attn_weights: Aufmerksamkeitsgewichte (batch, heads, seq, seq)
        """
        # ===== Mehrköpfige Aufmerksamkeit mit residualer Verbindung =====
        # Pre-Norm: Normalisiere VOR Aufmerksamkeit
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Residuale Verbindung
        
        # ===== Feedforward mit residualer Verbindung =====
        # Pre-Norm: Normalisiere VOR FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Residuale Verbindung
        
        return x, attn_weights


# Teste einen vollständigen Transformer-Block
block = TransformerBlock(
    d_model=768,
    num_heads=12,
    d_ff=3072,
    dropout=0.1
)

# Eingabe: Einbettungen aus Kapitel 9
embeddings_block = torch.randn(2, 6, 768)
mask_block = create_causal_mask(6)

# Vorwärtsdurchlauf
output_block, attn_weights_block = block(embeddings_block, mask_block)

print(f"Eingabe-Form:  {embeddings_block.shape}")  # Erwartet: (2, 6, 768)
print(f"Ausgabe-Form: {output_block.shape}")      # Erwartet: (2, 6, 768)
print(f"Aufmerksamkeitsgewichte Form: {attn_weights_block.shape}")  # Erwartet: (2, 12, 6, 6)

# Residuale überprüfen: Ausgabe sollte der Eingabe "ähnlich" sein (nicht völlig unterschiedlich)
print(f"\nEingabe-Mittelwert:  {embeddings_block.mean().item():.4f}")
print(f"Ausgabe-Mittelwert: {output_block.mean().item():.4f}")
print(f"Differenz:  {(output_block - embeddings_block).abs().mean().item():.4f}")
print("Differenz sollte moderat sein – nicht Null, nicht riesig")

## 6. Aufmerksamkeitsmuster visualisieren

In [ ]:
def visualize_attention(attn_weights, tokens, head_idx=0, ax=None):
    """
    Visualisiere Aufmerksamkeitsgewichte als Heatmap für einen bestimmten Kopf.
    
    Args:
        attn_weights: Aufmerksamkeitsgewichte (batch, heads, seq, seq)
        tokens: Liste von Token-Strings (Länge = seq)
        head_idx: Welcher Kopf zu visualisieren ist
        ax: Matplotlib-Achse (falls None, neue Figur erstellen)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    # Gewichte für angegebenen Kopf extrahieren (erstes Item im Batch)
    weights = attn_weights[0, head_idx].detach().cpu().numpy()
    
    # Heatmap plotten
    im = ax.imshow(weights, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    
    # Ticks und Labels setzen
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    
    # Labels
    ax.set_xlabel('Schlüssel (worauf geachtet wird)', fontsize=10)
    ax.set_ylabel('Abfrage (wer achtet)', fontsize=10)
    ax.set_title(f'Aufmerksamkeitsgewichte - Kopf {head_idx}', fontsize=12)
    
    # Farbbalken
    plt.colorbar(im, ax=ax, label='Aufmerksamkeitsgewicht')
    
    return ax


# Beispiel: Verarbeite einen echten Satz
from transformers import AutoTokenizer

# Tokenisiere
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "Der schnelle braune Fuchs springt über den faulen Hund"
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, seq)

# Token-Strings abrufen
tokens = [tokenizer.decode([t]) for t in token_ids[0]]
print(f"Tokens: {tokens}")

# Durch Einbettungsschicht laufen (simuliere Kapitel 9 Ausgabe)
embed_layer = nn.Embedding(50257, 768)
embeddings_viz = embed_layer(token_ids)  # (1, seq, 768)

# Kausale Maske erstellen
seq_len_viz = embeddings_viz.size(1)
mask_viz = create_causal_mask(seq_len_viz)

# Durch Transformer-Block laufen
block_viz = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_viz, attn_weights_viz = block_viz(embeddings_viz, mask_viz)

print(f"\nAufmerksamkeitsgewichte Form: {attn_weights_viz.shape}")  # Erwartet: (1, 12, seq, seq)

# Visualisiere erste 4 Köpfe
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, ax in enumerate(axes.flat):
    visualize_attention(attn_weights_viz, tokens, head_idx=i, ax=ax)

plt.tight_layout()
plt.show()

print("\nWas Sie sehen:")
print("1. Unteres Dreiecksmuster (kausale Maskierung funktioniert!)")
print("2. Unterschiedliche Muster pro Kopf (Vielfalt ist gut)")
print("3. Einige Köpfe fokussieren lokal, andere auf Langstrecken-Abhängigkeiten")

## 7. Die vollständige Pipeline

Verfolgen wir Daten von rohem Text bis zur Transformer-Block-Ausgabe:

In [ ]:
# ===== Kapitel 8: Tokenisierung =====
from transformers import AutoTokenizer
text = "Die Katze saß auf der Matte"  # Roher Text
tokenizer = AutoTokenizer.from_pretrained("gpt2")
token_ids = tokenizer.encode(text, return_tensors="pt")  # (1, 6)

print("Schritt 1: Tokenisierung")
print(f"Text: {text}")
print(f"Token-IDs: {token_ids}")
print(f"Form: {token_ids.shape}\n")

# ===== Kapitel 9: Einbettungen =====
# Simuliere GPT2Embeddings aus Kapitel 9
embed_layer = nn.Embedding(50257, 768)
embeddings = embed_layer(token_ids)  # (1, 6, 768)

print("Schritt 2: Einbettung")
print(f"Einbettungen Form: {embeddings.shape}")
print(f"Erstes Token Einbettung (erste 10 Dims): {embeddings[0, 0, :10]}\n")

# ===== Kapitel 10: Aufmerksamkeit (DIESES KAPITEL) =====
# Kausale Maske erstellen
seq_len_final = token_ids.size(1)  # 6
mask_final = create_causal_mask(seq_len_final)

# Transformer-Block
block_final = TransformerBlock(d_model=768, num_heads=12, d_ff=3072, dropout=0.1)
output_final, attn_weights_final = block_final(embeddings, mask_final)  # (1, 6, 768)

print("Schritt 3: Aufmerksamkeit (DIESES KAPITEL)")
print(f"Ausgabe-Form: {output_final.shape}")
print(f"Erstes Token nach Aufmerksamkeit (erste 10 Dims): {output_final[0, 0, :10]}\n")

print("Pipeline vollständig!")
print("Roher Text → Tokens → Einbettungen → Aufmerksamkeit → Kontextbewusste Vektoren")

print("\n===== Kapitel 11 Vorschau: 12 Blöcke stapeln =====")
print("In Kapitel 11 werden wir die Ausgabe durch 11 weitere Transformer-Blöcke leiten!")

## Übungen

### Übung 1: Manuelle Aufmerksamkeitsberechnung

Gegeben Q-, K-, V-Matrizen, berechnen Sie manuell Aufmerksamkeits-Scores, wenden Sie Softmax an und erhalten Sie die Ausgabe. Überprüfen Sie, ob Ihre Berechnungen mit `scaled_dot_product_attention()` übereinstimmen.

In [ ]:
# Erstelle einfache 2×3 Q, K, V (batch=1, seq=2, d_k=3)
Q_ex = torch.tensor([[[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
K_ex = torch.tensor([[[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]]])  # (1, 2, 3)
V_ex = torch.tensor([[[2.0, 0.0, 1.0], [1.0, 2.0, 0.0]]])  # (1, 2, 3)

# IHR CODE HIER:
# 1. Berechne scores = Q @ K^T
# 2. Skaliere durch sqrt(d_k)
# 3. Wende Softmax an
# 4. Gewichtete Summe mit V
# 5. Vergleiche mit scaled_dot_product_attention()

### Übung 2: Kausale Maskenüberprüfung

Erstellen Sie Aufmerksamkeitsgewichte mit und ohne kausale Maskierung. Überprüfen Sie, dass das obere Dreieck mit Maskierung Null ist.

In [ ]:
# IHR CODE HIER:
# 1. Erstelle Q, K, V für seq_len=5
# 2. Berechne Aufmerksamkeit OHNE Maske
# 3. Berechne Aufmerksamkeit MIT kausaler Maske
# 4. Drucke beide Aufmerksamkeitsgewichtsmatrizen
# 5. Überprüfe, dass oberes Dreieck in maskierter Version Null ist

### Übung 3: Multi-Head-Formen

Verfolgen Sie die Formtransformationen durch `MultiHeadAttention` mit spezifischen Zahlen.

In [ ]:
# IHR CODE HIER:
# Erstelle MHA mit d_model=512, num_heads=8, seq_len=10
# Drucke Form nach jedem Schritt:
# 1. Eingabe
# 2. Nach QKV-Projektion
# 3. Nach Umformung zur Trennung der Köpfe
# 4. Nach Aufmerksamkeitsberechnung
# 5. Nach Verkettung der Köpfe
# 6. Nach Ausgabeprojektion

### Übung 4: Einzel- vs. Multi-Head vergleichen

Führen Sie dieselbe Eingabe durch Einzelkopf- und 12-Kopf-Aufmerksamkeit. Vergleichen Sie Parameteranzahlen.

In [ ]:
# IHR CODE HIER:
# 1. Erstelle Einzelkopf-Aufmerksamkeit (d_model=768, num_heads=1)
# 2. Erstelle Multi-Head-Aufmerksamkeit (d_model=768, num_heads=12)
# 3. Zähle Parameter in jedem
# 4. Führe dieselbe Eingabe durch beide
# 5. Vergleiche Ausgaben und Parameteranzahlen

### Übung 5: Aufmerksamkeitsvisualisierung

Verarbeiten Sie Ihren eigenen Satz und visualisieren Sie verschiedene Aufmerksamkeitsköpfe.

In [ ]:
# IHR CODE HIER:
# 1. Wähle einen interessanten Satz (z.B. "Alice gab Bob ein Buch")
# 2. Tokenisiere ihn
# 3. Führe durch TransformerBlock
# 4. Visualisiere Köpfe 0, 3, 7, 11
# 5. Beschreibe, welche Muster Sie sehen (lokal vs. Langstrecke)

### Übung 6: Blöcke stapeln

Stapeln Sie 3 Transformer-Blöcke und verarbeiten Sie eine Sequenz durch alle.

In [ ]:
# IHR CODE HIER:
# 1. Erstelle 3 separate TransformerBlock-Instanzen
# 2. Leite Einbettungen durch block1 → block2 → block3
# 3. Drucke Form nach jedem Block
# 4. Vergleiche Eingabe-Einbettungen mit finaler Ausgabe
# 5. Wie unterschiedlich sind sie?

### Übung 7: Pre-Norm vs. Post-Norm

Implementieren Sie einen Post-Norm-Transformer-Block und vergleichen Sie mit Pre-Norm.

In [ ]:
# IHR CODE HIER:
# 1. Implementiere TransformerBlockPostNorm, wo LayerNorm NACH kommt
# 2. Führe dieselbe Eingabe durch Pre-Norm und Post-Norm
# 3. Vergleiche Ausgaben
# 4. Welcher hat stabilere Gradienten? (Sie können Gradientengrößen prüfen)

### Übung 8: Parameteranzahl

Berechnen Sie die exakte Parameteranzahl für einen Transformer-Block.

In [ ]:
# IHR CODE HIER:
# Für d_model=768, num_heads=12, d_ff=3072:
# 1. Zähle QKV-Projektionsparameter
# 2. Zähle Ausgabeprojektionsparameter
# 3. Zähle FFN-Parameter (fc1 + fc2)
# 4. Zähle LayerNorm-Parameter (2 Schichten)
# 5. Summiere für Gesamt
# 6. Überprüfe mit model.parameters()

## Kapitelzusammenfassung

**Was wir gebaut haben:**

1. Skalierte Punktprodukt-Aufmerksamkeit (Q, K, V → Scores → Softmax → gewichtete Summe)
2. Kausale Maskierung für autoregressive Generierung (kein Spicken in die Zukunft)
3. Effiziente mehrköpfige Aufmerksamkeit (1 Kopf → 12 Köpfe via Umformungstricks)
4. Vollständige Transformer-Blöcke (Aufmerksamkeit + FFN + residuale Verbindungen + Schichtnormalisierung)
5. Aufmerksamkeitsvisualisierung (Heatmaps zeigen, worauf das Modell achtet)

**Kernkonzepte:**

- **Statische Einbettungen** (Kap. 9) → **Kontextbewusste Repräsentationen** (Kap. 10)
- **Query/Key/Value**: Drei gelernte Projektionen mit unterschiedlichen Rollen
- **Aufmerksamkeitsgewichte**: Softmax-Wahrscheinlichkeiten, die Relevanz zeigen
- **Multi-Head**: Verschiedene Köpfe lernen verschiedene Muster (keine zusätzlichen Parameter!)
- **Residuale Verbindungen + Normalisierung**: Ermöglichen tiefe Netzwerke (100+ Schichten)

**Nächstes:** Kapitel 11 wird diese Blöcke stapeln und den Language-Modeling-Kopf hinzufügen, um ein vollständiges GPT-Modell zu erstellen!